1. Setup & Imports

In [15]:
print("Gerekli kütüphaneler içe aktarılıyor: pandas, numpy, sklearn.")
# basic
import pandas as pd
import numpy as np

# text processing
from sklearn.feature_extraction.text import TfidfVectorizer

# similarity
from sklearn.metrics.pairwise import cosine_similarity

Gerekli kütüphaneler içe aktarılıyor: pandas, numpy, sklearn.


2. Data Loading

In [16]:
print("Ürün ve inceleme verileri URL'lerden DataFrame'lere yükleniyor.")
products_url = "https://raw.githubusercontent.com/bengubal/cosmetic-dupe-finder/refs/heads/main/data/cleaned_products.csv"
reviews_url = "https://raw.githubusercontent.com/bengubal/cosmetic-dupe-finder/refs/heads/main/data/cleaned_reviews.csv"

df_products = pd.read_csv(products_url)
df_reviews = pd.read_csv(reviews_url)

Ürün ve inceleme verileri URL'lerden DataFrame'lere yükleniyor.


3. Data Inspection

In [17]:
print("Veri setlerinin ilk beş satırı ve ürün veri setinin genel bilgileri gösteriliyor.")
df_products.head()
df_products.info()

df_reviews.head()

Veri setlerinin ilk beş satırı ve ürün veri setinin genel bilgileri gösteriliyor.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          2999 non-null   object 
 1   product_name        2999 non-null   object 
 2   brand_name          2999 non-null   object 
 3   ingredients         2999 non-null   object 
 4   price_usd           2999 non-null   float64
 5   rating              2999 non-null   float64
 6   reviews             2999 non-null   float64
 7   primary_category    2999 non-null   object 
 8   secondary_category  2999 non-null   object 
 9   tertiary_category   2934 non-null   object 
 10  clean_ingredients   2999 non-null   object 
dtypes: float64(3), object(8)
memory usage: 257.9+ KB


,product_id,review_text,rating,is_recommended,helpfulness,clean_review_text
0,P504322,I use this with the Nudestix “Citrus Clean Bal...,5,1.0,1.0,i use this with the nudestix citrus clean balm...
1,P420652,I bought this lip mask after reading the revie...,1,0.0,NaN,i bought this lip mask after reading the revie...
2,P420652,My review title says it all! I get so excited ...,5,1.0,NaN,my review title says it all i get so excited t...
3,P420652,I’ve always loved this formula for a long time...,5,1.0,NaN,i ve always loved this formula for a long time...
4,P420652,"If you have dry cracked lips, this is a must h...",5,1.0,NaN,if you have dry cracked lips this is a must ha...


4. Text Preparation

In [18]:
print("Ürün açıklamasını ve bileşenleri birleştiren 'text' sütunu oluşturuluyor.")
df_products['text'] = (
    df_products['clean_ingredients'].fillna('') + " " +
    df_products['product_name'].fillna('')
)

Ürün açıklamasını ve bileşenleri birleştiren 'text' sütunu oluşturuluyor.


5. TF-IDF Matrix

In [19]:
print("TF-IDF Vektörleyici başlatılıyor ve ürün metinlerinden TF-IDF matrisi oluşturuluyor.")
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

tfidf_matrix = tfidf.fit_transform(df_products['text'])

TF-IDF Vektörleyici başlatılıyor ve ürün metinlerinden TF-IDF matrisi oluşturuluyor.


6. Similarity Calculation

In [20]:
print("TF-IDF matrisi kullanılarak ürünler arasındaki kosinüs benzerliği hesaplanıyor.")
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

TF-IDF matrisi kullanılarak ürünler arasındaki kosinüs benzerliği hesaplanıyor.


7. Recommendation Function

In [21]:
print("Benzer ürünleri öneren 'recommend' fonksiyonu tanımlanıyor.")
def recommend(product_id, top_n=5):
    idx = df_products[df_products['product_id'] == product_id].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    product_indices = [i[0] for i in sim_scores]

    return df_products.iloc[product_indices][
        ['product_id', 'product_name', 'price_usd']
    ]

Benzer ürünleri öneren 'recommend' fonksiyonu tanımlanıyor.


8. Price Filtering

In [22]:
print("Fiyat filtrelemesi ile benzer ürünleri öneren 'recommend_with_price' fonksiyonu tanımlanıyor.")
def recommend_with_price(product_id, top_n=5, max_price=None):
    idx = df_products[df_products['product_id'] == product_id].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n*5]

    product_indices = [i[0] for i in sim_scores]
    candidates = df_products.iloc[product_indices]

    if max_price:
        candidates = candidates[candidates['price_usd'] <= max_price]

    return candidates.head(top_n)[
        ['product_id', 'product_name', 'price_usd']
    ]

Fiyat filtrelemesi ile benzer ürünleri öneren 'recommend_with_price' fonksiyonu tanımlanıyor.


9. Test Outputs

In [23]:
print("'recommend' fonksiyonu örnek bir ürün kimliği ile test ediliyor.")
sample_id = df_products['product_id'].iloc[0]

recommend(sample_id)

'recommend' fonksiyonu örnek bir ürün kimliği ile test ediliyor.


,product_id,product_name,price_usd
2,P457233,Baomint Leave In Conditioning Styler,26.0
4,P457234,Baomint Moisturizing Shampoo,24.0
10,P457235,Baomint Moisturizing Curl Defining Gel,32.0
8,P457236,Baomint Moisturizing Curl Defining Cream,32.0
7,P457237,Baomint Protect + Shine Oil Blend,24.0


In [24]:
print("'recommend_with_price' fonksiyonu örnek bir ürün kimliği ve maksimum fiyat ile test ediliyor.")
recommend_with_price(sample_id, max_price=300)

'recommend_with_price' fonksiyonu örnek bir ürün kimliği ve maksimum fiyat ile test ediliyor.


,product_id,product_name,price_usd
2,P457233,Baomint Leave In Conditioning Styler,26.0
4,P457234,Baomint Moisturizing Shampoo,24.0
10,P457235,Baomint Moisturizing Curl Defining Gel,32.0
8,P457236,Baomint Moisturizing Curl Defining Cream,32.0
7,P457237,Baomint Protect + Shine Oil Blend,24.0


10. Brand Filtering (Diversity Improvement)

In [38]:
print("df_products DataFrame'inin sütun adları görüntüleniyor.")
df_products.columns

df_products DataFrame'inin sütun adları görüntüleniyor.


Index(['product_id', 'product_name', 'brand_name', 'ingredients', 'price_usd',
       'rating', 'reviews', 'primary_category', 'secondary_category',
       'tertiary_category', 'clean_ingredients', 'text'],
      dtype='object')

In [39]:
print("Marka adları temizleniyor ve küçük harfe dönüştürülüyor.")
df_products['brand_name'] = df_products['brand_name'].fillna('').str.lower().str.strip()

Marka adları temizleniyor ve küçük harfe dönüştürülüyor.


In [40]:
print("Aynı marka ürünleri hariç tutarak öneri yapan 'recommend_no_same_brand' fonksiyonu tanımlanıyor.")
def recommend_no_same_brand(product_id, top_n=5):
    idx = df_products[df_products['product_id'] == product_id].index[0]

    target_brand = df_products.loc[idx, 'brand_name']

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # daha geniş aday havuzu
    sim_scores = sim_scores[1:top_n*10]

    product_indices = [i[0] for i in sim_scores]
    candidates = df_products.iloc[product_indices]

    # ❗ aynı markayı çıkar
    candidates = candidates[candidates['brand_name'] != target_brand]

    return candidates.head(top_n)[
        ['product_id', 'product_name', 'brand_name', 'price_usd']
    ]

Aynı marka ürünleri hariç tutarak öneri yapan 'recommend_no_same_brand' fonksiyonu tanımlanıyor.


In [41]:
print("Fiyat filtrelemesi ve aynı marka hariç tutularak alternatif ürünler öneren 'recommend_alternatives' fonksiyonu tanımlanıyor.")
def recommend_alternatives(product_id, top_n=5, max_price=None):
    idx = df_products[df_products['product_id'] == product_id].index[0]

    target_brand = df_products.loc[idx, 'brand_name']
    target_price = df_products.loc[idx, 'price_usd']

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n*15]

    product_indices = [i[0] for i in sim_scores]
    candidates = df_products.iloc[product_indices]

    # marka filtresi
    candidates = candidates[candidates['brand_name'] != target_brand]

    # fiyat filtresi
    if max_price:
        candidates = candidates[candidates['price_usd'] <= max_price]
    else:
        candidates = candidates[candidates['price_usd'] <= target_price]

    return candidates.head(top_n)[
        ['product_id', 'product_name', 'brand_name', 'price_usd']
    ]

Fiyat filtrelemesi ve aynı marka hariç tutularak alternatif ürünler öneren 'recommend_alternatives' fonksiyonu tanımlanıyor.


In [42]:
print("'recommend_alternatives' fonksiyonu örnek bir ürün kimliği ile test ediliyor.")
sample_id = df_products['product_id'].iloc[0]

print("ORİJİNAL ÜRÜN:")
print(df_products[df_products['product_id'] == sample_id][
    ['product_name', 'brand_name', 'price_usd']
])

print("\nÖNERİLER:")
recommend_alternatives(sample_id)

'recommend_alternatives' fonksiyonu örnek bir ürün kimliği ile test ediliyor.
ORİJİNAL ÜRÜN:
                          product_name    brand_name  price_usd
0  Baomint Deep Conditioning Treatment  adwoa beauty       39.0

ÖNERİLER:


,product_id,product_name,brand_name,price_usd
1757,P500477,Plumping Deep Conditioner,melanin haircare,32.00
223,P500714,Elixir Hair Oil Treatment with Castor Oil,bondiboost,19.95
249,P388628,"Don't Despair, Repair! Deep Conditioning Hair ...",briogeo,39.00
237,P504004,Heavenly Hydration Pre-Shampoo Hair Oil,bondiboost,29.99
656,P457411,Deep Conditioning Hair Treatment,dae,28.00
